## 对称量化+正确率测试(提取test_vector.h中的量化参数用于测试)

In [1]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import re

# ==========================================
# 工具函数：从 C 头文件中暴力解析参数和数组
# ==========================================
def parse_header_file(filepath):
    """从 test_vector.h 中解析出所需的宏参数和一维数组"""
    with open(filepath, 'r') as f:
        content = f.read()

    # 1. 解析 s_in (假设你已经在头文件中加上了 // Global_Scale: s_in=...)
    # 如果没加，请手动在这里填入你校准时打印出的 s_in 值
    s_in_match = re.search(r's_in=([0-9.]+)', content)
    s_in = float(s_in_match.group(1)) if s_in_match else 0.007874 # 填入你的默认值兜底

    # 2. 解析 Mult 和 Shift
    # 使用正则匹配 // L1: Mult=xxx, Shift=xxx 这种格式
    l1_match = re.search(r'L1: Mult=(\d+), Shift=(\d+)', content)
    l2_match = re.search(r'L2: Mult=(\d+), Shift=(\d+)', content)
    fc_match = re.search(r'FC: Mult=(\d+), Shift=(\d+)', content)

    params = {
        's_in': s_in,
        'mult1': int(l1_match.group(1)), 'shift1': int(l1_match.group(2)),
        'mult2': int(l2_match.group(1)), 'shift2': int(l2_match.group(2)),
        'mult_fc': int(fc_match.group(1)), 'shift_fc': int(fc_match.group(2)),
    }

    # 3. 解析 C 数组提取纯数字
    def extract_array(array_name, dtype=torch.int8):
        # 寻找类似于 const int8_t test_conv1_weight[432] = { ... }; 的块
        pattern = r'{}\[\d+\]\s*=\s*{{([^}}]+)}}'.format(array_name)
        match = re.search(pattern, content)
        if not match:
            raise ValueError(f"找不到数组 {array_name}")
        
        # 将 "1, 2, -3, \n 4" 这种字符串切分成纯数字列表
        num_strs = match.group(1).replace('\n', '').split(',')
        nums = [int(n.strip()) for n in num_strs if n.strip()]
        return torch.tensor(nums, dtype=dtype)

    # 提取并重塑为网络计算所需的维度 (极度关键)
    params['w1_q'] = extract_array('test_conv1_weight').view(16, 3, 3, 3)
    params['b1_q'] = extract_array('test_conv1_bias', torch.int32)
    
    params['w2_q'] = extract_array('test_conv2_weight').view(32, 16, 3, 3)
    params['b2_q'] = extract_array('test_conv2_bias', torch.int32)
    
    params['wfc_q'] = extract_array('test_fc_weight').view(10, 2048)
    params['bfc_q'] = extract_array('test_fc_bias', torch.int32)

    return params


# ==========================================
# 主流程：在全量测试集上验证纯 INT8 逻辑
# ==========================================
def main():
    print(">>> 步骤 1/3: 加载 10000 张测试集...")
    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)
    
    print(">>> 步骤 2/3: 从 test_vector.h 加载 NPU 量化参数...")
    try:
        hw_params = parse_header_file('test_vector.h')
    except Exception as e:
        print(f"解析头文件失败: {e}。请确保头文件格式正确且包含 s_in。")
        return

    s_in = hw_params['s_in']
    w1_q, b1_q = hw_params['w1_q'], hw_params['b1_q']
    mult1, shift1 = hw_params['mult1'], hw_params['shift1']
    
    w2_q, b2_q = hw_params['w2_q'], hw_params['b2_q']
    mult2, shift2 = hw_params['mult2'], hw_params['shift2']
    
    wfc_q, bfc_q = hw_params['wfc_q'], hw_params['bfc_q']
    mult_fc, shift_fc = hw_params['mult_fc'], hw_params['shift_fc']

    print("\n>>> 步骤 3/3: 在 10000 张测试集上运行纯 INT8 验证...")
    correct = 0
    total = 0

    # 整个循环内绝对不出现任何基于浮点的非线性计算
    for images, labels in testloader:
        # 1. 输入量化 (严格使用全局 s_in)
        img_q = torch.round(images / s_in)
        img_q = torch.clamp(img_q, -127, 127).to(torch.int8)

        # 2. 模拟 NPU Conv1 -> ReLU -> Pool1
        mac1 = F.conv2d(img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
        q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
        q_pool1_out = F.max_pool2d(F.relu(q_conv1_out.float()), 2).to(torch.int8)

        # 3. 模拟 NPU Conv2 -> ReLU -> Pool2
        mac2 = F.conv2d(q_pool1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
        q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
        q_pool2_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)

        # 4. 模拟 NPU FC 阶段
        q_flat_in = q_pool2_out.view(q_pool2_out.size(0), -1) 
        mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
        q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

        # 5. 统计正确率
        _, predicted = torch.max(q_fc_out.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"\n✅ INT8 硬件级量化模型在 10000 张测试图上的最终准确率: {accuracy:.2f}%")
    print(f"(如果这个数值与你的 FP32 原始准确率相差在 1%-2% 以内，说明量化策略大获成功！)")

if __name__ == "__main__":
    main()

>>> 步骤 1/3: 加载 10000 张测试集...
Files already downloaded and verified
>>> 步骤 2/3: 从 test_vector.h 加载 NPU 量化参数...

>>> 步骤 3/3: 在 10000 张测试集上运行纯 INT8 验证...

✅ INT8 硬件级量化模型在 10000 张测试图上的最终准确率: 68.99%
(如果这个数值与你的 FP32 原始准确率相差在 1%-2% 以内，说明量化策略大获成功！)
